In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('/content/Student Social Media And Mental Health Impact.csv')

In [3]:
len(df.columns)

13

In [4]:
df.columns

Index(['Age', 'Gender', 'Country', 'Academic_Level', 'Most_Used_Platform',
       'Purpose_Of_Use', 'Avg_Daily_Usage_Hours', 'Daily_Unlocks',
       'Study_Hours', 'Physical_Activity_Hours', 'Sleep_Hours_Per_Night',
       'Stress_Level', 'Mental_Health_Score'],
      dtype='object')

In [5]:
df.shape

(5000, 13)

In [6]:
df.isnull().sum()

,0
Age,0
Gender,0
Country,0
Academic_Level,0
Most_Used_Platform,0
Purpose_Of_Use,0
Avg_Daily_Usage_Hours,0
Daily_Unlocks,0
Study_Hours,0
Physical_Activity_Hours,0


In [7]:
df.duplicated().sum()

np.int64(2)

In [8]:
top_countries  = df['Country'].value_counts().index[:10].to_list()

In [9]:
def group_countries(Country):
  if Country in top_countries:
    return Country
  else:
    return 'other'

In [10]:
df['grouped_country']=df['Country'].apply(group_countries)

In [11]:
df['grouped_country'].value_counts()

,count
grouped_country,
Other,1880
other,1352
India,389
USA,355
Canada,230
Australia,198
UK,185
Germany,136
Mexico,94


In [12]:
num_cols = df.select_dtypes(include='number')
num_cols.skew()

,0
Age,0.155325
Avg_Daily_Usage_Hours,0.005606
Daily_Unlocks,0.002351
Study_Hours,0.435819
Physical_Activity_Hours,0.039971
Sleep_Hours_Per_Night,0.124050
Mental_Health_Score,0.206567


In [13]:
skew_cols = ['Study_Hours']

In [14]:
from sklearn.model_selection import train_test_split

skwewd_col         = ['Study_Hours']
other_numeric_cols = ["Age", "Avg_Daily_Usage_Hours", "Daily_Unlocks","Physical_Activity_Hours",
                      "Sleep_Hours_Per_Night"]
ordinal_col        = ['Stress_Level']
normal_col         = ["Gender", "Academic_Level", "Most_Used_Platform", "Purpose_Of_Use", "grouped_country"]


feature_col = skwewd_col + other_numeric_cols + ordinal_col + normal_col
X = df[feature_col]
y = df['Mental_Health_Score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

In [21]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    StandardScaler,
    OrdinalEncoder,
    OneHotEncoder
)
from sklearn.compose import ColumnTransformer
import numpy as np

# Skewed numerical columns
skew_pipeline = Pipeline(steps=[
    ('log_transform', FunctionTransformer(np.log1p)),
    ('scaler', StandardScaler())
])

# Other numerical columns
plain_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Ordinal columns
ordinal_pipeline = Pipeline(steps=[
    ('encode', OrdinalEncoder(
        categories=[['Low', 'Medium', 'High', 'Very High']]
    ))
])

# Nominal categorical columns
nominal_pipeline = Pipeline(steps=[
    ('encode', OneHotEncoder(handle_unknown='ignore'))
])

# Combine all preprocessing pipelines
preprocessor = ColumnTransformer(transformers=[
    ('Skewed_pipeline', skew_pipeline, skwewd_col),
    ('Ordinal', ordinal_pipeline, ordinal_col),
    ('Plain_numeric', plain_pipeline, other_numeric_cols),
    ('Normal', nominal_pipeline, normal_col)
])

In [25]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error,r2_score,mean_absolute_error
lr_pipeline = Pipeline(steps= [('preprocessor',preprocessor),('regressor',LinearRegression())])
lr_pipeline.fit(X_train,y_train)
lr_pred = lr_pipeline.predict(X_test)
lr_pred_train  = lr_pipeline.predict(X_train)
lr_r2_testing  = r2_score(y_test,lr_pred)
lr_r2_training = r2_score(y_train,lr_pred_train)
lr_mae         = mean_absolute_error(y_test,lr_pred)
print(f"Accuracy of trainig is {lr_r2_training}")
print(f"Accuracy of testing is {lr_r2_testing }")
print(f"Accuracy of MAE is {lr_mae }")

Accuracy of trainig is 0.7254266761392683
Accuracy of testing is 0.7597338655764386
Accuracy of MAE is 0.5085876308254499


In [26]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor

rf_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('random_forest', RandomForestRegressor(random_state=42))
])
rf_pipeline.fit(X_train,y_train)
rf_pred = rf_pipeline.predict(X_test)
rf_pred2 = rf_pipeline.predict(X_train)

rf_r2_score = r2_score(y_train,rf_pred2)
rf_r2_score2 = r2_score(y_test,rf_pred)
rf_mae  = mean_absolute_error(y_test,rf_pred)
print(rf_r2_score)
print(rf_r2_score2)
print(rf_mae)

0.9827992697060809
0.8916348621211099
0.3219965111111112


In [34]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {'random_forest__n_estimators' : [100,200,300],
              'random_forest__max_depth' : [5,10,15],
              'random_forest__min_samples_split': [2,5,10],
              'random_forest__min_samples_leaf' : [1,2,4]
              }
random_search = RandomizedSearchCV(
    estimator = rf_pipeline,
    param_distributions = param_grid,
    n_iter = 15,
    cv = 5,
    scoring = 'r2',
    random_state = 42,
    # n jobs ka matlab hoota ha sara processor ko laga to kam par
    n_jobs = -1

)
random_search.fit(X_train,y_train)

RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('preprocessing',
                                              ColumnTransformer(transformers=[('Skewed_pipeline',
                                                                               Pipeline(steps=[('log_transform',
                                                                                                FunctionTransformer(func=<ufunc 'log1p'>)),
                                                                                               ('scaler',
                                                                                                StandardScaler())]),
                                                                               ['Study_Hours']),
                                                                              ('Ordinal',
                                                                               Pipeline(steps=[('encode',
                                                                                                OrdinalEncoder(categories=[['Low',
                                                                                                                            'Medium',
                                                                                                                            'High',
                                                                                                                            'Very '
                                                                                                                            'High']]))]),
                                                                               ['St...
                                                                                'Most_Used_Platform',
                                                                                'Purpose_Of_Use',
                                                                                'grouped_country'])])),
                                             ('random_forest',
                                              RandomForestRegressor(random_state=42))]),
                   n_iter=15, n_jobs=-1,
                   param_distributions={'random_forest__max_depth': [5, 10, 15],
                                        'random_forest__min_samples_leaf': [1,
                                                                            2,
                                                                            4],
                                        'random_forest__min_samples_split': [2,
                                                                             5,
                                                                             10],
                                        'random_forest__n_estimators': [100,
                                                                        200,
                                                                        300]},
                   random_state=42, scoring='r2')

In [35]:
random_search.best_params_

{'random_forest__n_estimators': 200,
 'random_forest__min_samples_split': 5,
 'random_forest__min_samples_leaf': 2,
 'random_forest__max_depth': 15}

In [36]:
rf_best_pipeline = random_search.best_estimator_
rf_best_preds    = rf_best_pipeline.predict(X_test)

print(f"R2 Score for random forest after hyperparameter tuning {r2_score(y_test, rf_best_preds)}")
print(f"MAE Score for random forest after hyperparameter tuning {mean_absolute_error(y_test, rf_best_preds)}")

R2 Score for random forest after hyperparameter tuning 0.8794776137277109
MAE Score for random forest after hyperparameter tuning 0.34318189547702266


In [ ]:
from sklearn.metrics import mean_squared_error

# Calculate RMSE for Linear Regression
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))

# Calculate RMSE for default Random Forest
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))

# Calculate RMSE for tuned Random Forest
rf_tuned_preds          = random_search.best_estimator_.predict(X_test)
rf_tuned_rmse           = np.sqrt(mean_squared_error(y_test, rf_tuned_preds))
rf_tuned_training_preds = random_search.best_estimator_.predict(X_train)
r2_tuned_training       = r2_score(y_train, rf_tuned_training_preds)
rf_tuned_mae            = mean_absolute_error(y_test, rf_tuned_preds)
rf_tuned_r2             = r2_score(y_test, rf_tuned_preds)


# Create a DataFrame to consolidate results
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest (default)', 'Random Forest (tuned)'],
    'R2': [lr_r2_testing, rf_r2_testing, rf_tuned_r2],
    'Training R2': [lr_r2_training, rf_r2_training, r2_tuned_training],
    'MAE': [lr_mae, rf_mae, rf_tuned_mae],
    'RMSE': [lr_rmse, rf_rmse, rf_tuned_rmse]
})

print(results)


In [37]:
import joblib

joblib.dump(rf_pipeline,'Mental_health_model.pkl')

['Mental_health_model.pkl']